In [ ]:
# =================================================================
# INSTALACIÓN Y CONFIGURACIÓN INICIAL
# =================================================================

!pip install gnews pandas nltk

import pandas as pd
import os
from gnews import GNews
from google.colab import drive
from datetime import datetime

# Montar Google Drive
drive.mount('/content/drive')

# CONFIGURACIÓN
path_proyecto = "/content/drive/MyDrive/TitulacionF"

print("✅ Librerías instaladas y Drive montado")
print(f"📁 Ruta del proyecto: {path_proyecto}")



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Librerías instaladas y Drive montado
📁 Ruta del proyecto: /content/drive/MyDrive/TitulacionF


In [ ]:
# =================================================================
# CONFIGURACIÓN DE BÚSQUEDAS EXPANDIDAS
# =================================================================

# Cantones de Manabí (principales)
cantones = [
    "Portoviejo", "Manta", "Chone", "Montecristi", "Jipijapa",
    "Paján", "Sucre", "Bolívar", "Pedernales", "Tosagua",
    "Rocafuerte", "Santa Ana", "El Carmen", "Calceta",
    "Bahía de Caráquez", "San Vicente"
]

# Sectores económicos ampliados
sectores_economia = [
    # Sectores productivos
    "economía", "producción", "industria", "manufactura",

    # Empleo y trabajo
    "empleo", "desempleo", "trabajo", "empleos", "contratación",
    "despidos", "subempleo",

    # Comercio y negocios
    "comercio", "ventas", "negocios", "emprendimiento", "pymes",
    "microempresas", "pequeñas empresas", "comerciantes",

    # Sectores específicos
    "turismo", "hotelería", "restaurantes", "gastronomía",
    "atún", "pesca", "camarón", "acuacultura", "exportación pesquera",
    "agricultura", "cultivos", "café", "cacao", "banano", "maíz",
    "ganado", "ganadería", "lechería",

    # Construcción e infraestructura
    "construcción", "vivienda", "inmobiliario", "obras",

    # Finanzas
    "inversión", "financiamiento", "créditos", "bancos",
    "remesas", "dinero",

    # Problemáticas
    "crisis", "pobreza", "inflación", "carestía", "precios",
    "escasez", "cierre de negocios", "quiebra",

    # Desarrollo
    "desarrollo", "crecimiento económico", "recuperación económica",
    "reactivación", "inversión pública", "proyectos"
]

# Términos complementarios para Manabí en general
terminos_generales_manabi = [
    "economía Manabí",
    "situación económica Manabí",
    "desarrollo Manabí",
    "crisis Manabí",
    "inversión Manabí",
    "pobreza Manabí",
    "desempleo Manabí",
    "comercio Manabí",
    "exportaciones Manabí",
    "turismo Manabí",
    "agricultura Manabí",
    "pesca Manabí",
    "infraestructura Manabí",
    "zona afectada Manabí",  # Por terremotos
    "reconstrucción Manabí",
    "puerto Manta",
    "zona franca Manta",
    "ZEDE Manta"  # Zona Especial de Desarrollo Económico
]

print(f"Configuración de búsquedas:")
print(f"  • Cantones: {len(cantones)}")
print(f"  • Sectores económicos: {len(sectores_economia)}")
print(f"  • Términos generales: {len(terminos_generales_manabi)}")

Configuración de búsquedas:
  • Cantones: 16
  • Sectores económicos: 61
  • Términos generales: 18


In [ ]:
# =================================================================
# GENERACIÓN DE CONSULTAS COMBINADAS
# =================================================================

# Generar consultas: cantones × sectores
consultas_combinadas = []
for canton in cantones:
    for sector in sectores_economia:
        consultas_combinadas.append(f"{sector} {canton}")

# Agregar términos generales de Manabí
consultas_totales = consultas_combinadas + terminos_generales_manabi

print(f"\n🔍 Total de consultas a realizar: {len(consultas_totales)}")
print(f"  • Consultas combinadas (cantón × sector): {len(consultas_combinadas)}")
print(f"  • Consultas generales Manabí: {len(terminos_generales_manabi)}")



🔍 Total de consultas a realizar: 994
  • Consultas combinadas (cantón × sector): 976
  • Consultas generales Manabí: 18


In [ ]:
# =================================================================
# FUNCIÓN DE RECOLECCIÓN
# =================================================================

def recoleccion_5_años_limpia():
    """
    Recolección masiva de noticias de 6 años
    Retorna DataFrame con todos los resultados sin duplicados
    """

    print("🚀 Iniciando recolección masiva de 6 años...\n")
    print("=" * 60)
    fecha_inicio = datetime(2020, 1, 1)
    fecha_fin = datetime.now()

    # Configurar scraper con 6 años y español de Ecuador
    scraper = GNews(
        language='es',
        country='EC',
        #period='6y',  # 6 años
        start_date=fecha_inicio,
        end_date=fecha_fin,
        max_results=100  # Máximo por consulta
    )

    resultados = []
    errores = 0

    for idx, consulta in enumerate(consultas_totales, 1):
        # Mostrar progreso cada 20 consultas
        if idx % 20 == 0:
            print(f"\n[{idx}/{len(consultas_totales)}] Procesadas {idx} consultas...")
            print(f"📊 Resultados hasta ahora: {len(resultados)}")

        try:
            noticias = scraper.get_news(consulta)

            for noticia in noticias:
                resultados.append({
                    'fecha': noticia['published date'],
                    'texto': noticia['title'] + ". " + noticia['description'],
                    'url': noticia['url'],
                    'fuente': noticia['publisher']['title'],
                    'consulta': consulta  # Para tracking
                })

        except Exception as e:
            errores += 1
            if errores <= 5:  # Mostrar solo primeros 5 errores
                print(f"⚠️ Error en '{consulta[:30]}...': {str(e)[:50]}")
            continue

    print("\n" + "=" * 60)
    print(f"✅ Recolección completada")
    print(f"  • Consultas procesadas: {len(consultas_totales)}")
    print(f"  • Errores encontrados: {errores}")
    print(f"  • Noticias obtenidas (con duplicados): {len(resultados)}")

    if resultados:
        # Crear DataFrame y eliminar duplicados
        df = pd.DataFrame(resultados)
        df_unico = df.drop_duplicates(subset=['url'])

        duplicados = len(df) - len(df_unico)
        print(f"  • Duplicados eliminados: {duplicados}")
        print(f"  • Registros únicos finales: {len(df_unico)}")

        return df_unico
    else:
        print("❌ No se obtuvieron resultados")
        return pd.DataFrame()

In [ ]:
# =================================================================
# EJECUTAR RECOLECCIÓN Y GUARDAR
# =================================================================

# Ejecutar scraping
df_noticias = recoleccion_5_años_limpia()

# Guardar en Drive
if not df_noticias.empty:
    ruta_csv = f"{path_proyecto}/datos_economicos_manabi.csv"
    df_noticias.to_csv(ruta_csv, index=False, encoding='utf-8-sig')

    print(f"\n💾 Datos guardados exitosamente")
    print(f"📁 Ubicación: datos_economicos_manabi.csv")
    print(f"📊 Total de registros: {len(df_noticias)}")

    # Mostrar estadísticas
    print("\n📈 ESTADÍSTICAS DEL DATASET:")
    print("-" * 60)

    # Top 10 fuentes más frecuentes
    print("\n🗞️ Top 10 Fuentes:")
    top_fuentes = df_noticias['fuente'].value_counts().head(10)
    for fuente, count in top_fuentes.items():
        print(f"  • {fuente}: {count} noticias")

    # Distribución temporal
    df_noticias['año'] = pd.to_datetime(df_noticias['fecha']).dt.year
    print("\n📅 Distribución por año:")
    dist_anual = df_noticias['año'].value_counts().sort_index()
    for año, count in dist_anual.items():
        print(f"  • {año}: {count} noticias")

    # Ejemplos de noticias
    print("\n📰 EJEMPLOS DE NOTICIAS RECOLECTADAS:")
    print("-" * 60)
    for idx, row in df_noticias.head(5).iterrows():
        print(f"\n[{row['fuente']}] - {row['fecha']}")
        print(f"{row['texto'][:150]}...")
else:
    print("\n❌ No se guardaron datos. Verifica la configuración.")

🚀 Iniciando recolección masiva de 6 años...


[20/994] Procesadas 20 consultas...
📊 Resultados hasta ahora: 727

[40/994] Procesadas 40 consultas...
📊 Resultados hasta ahora: 1360

[60/994] Procesadas 60 consultas...
📊 Resultados hasta ahora: 2136

[80/994] Procesadas 80 consultas...
📊 Resultados hasta ahora: 3253

[100/994] Procesadas 100 consultas...
📊 Resultados hasta ahora: 4378

[120/994] Procesadas 120 consultas...
📊 Resultados hasta ahora: 5604

[140/994] Procesadas 140 consultas...
📊 Resultados hasta ahora: 6269

[160/994] Procesadas 160 consultas...
📊 Resultados hasta ahora: 6655

[180/994] Procesadas 180 consultas...
📊 Resultados hasta ahora: 7243

[200/994] Procesadas 200 consultas...
📊 Resultados hasta ahora: 7955

[220/994] Procesadas 220 consultas...
📊 Resultados hasta ahora: 8512

[240/994] Procesadas 240 consultas...
📊 Resultados hasta ahora: 9277

[260/994] Procesadas 260 consultas...
📊 Resultados hasta ahora: 9737

[280/994] Procesadas 280 consultas...
📊 Resultados ha

/tmp/ipykernel_311/3480187866.py:28: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_noticias['año'] = pd.to_datetime(df_noticias['fecha']).dt.year


In [ ]:
# =================================================================
# LIMPIEZA DE TEXT
# =================================================================

import re
import nltk
from nltk.corpus import stopwords

# Descargar recursos de NLTK
nltk.download('stopwords', quiet=True)
stop_words = set(stopwords.words('spanish'))

# Agregar stopwords personalizadas para noticias
stop_words_custom = stop_words.union({
    'dijo', 'señaló', 'afirmó', 'indicó', 'explicó', 'aseguró',
    'según', 'través', 'partir', 'realizar', 'hacer',
    'año', 'años', 'día', 'días', 'mes', 'meses'
})

def limpiar_texto_mejorado(texto):
    """
    Limpieza mejorada de texto con más filtros
    """
    texto = str(texto).lower()

    # Eliminar URLs
    texto = re.sub(r'http\S+|www\S+|https\S+', '', texto, flags=re.MULTILINE)

    # Eliminar emails
    texto = re.sub(r'\S+@\S+', '', texto)

    # Eliminar menciones y hashtags
    texto = re.sub(r'@\w+|#\w+', '', texto)

    # Eliminar números solos (pero mantener palabras con números)
    texto = re.sub(r'\b\d+\b', '', texto)

    # Eliminar caracteres especiales y puntuación
    texto = re.sub(r'[^\w\s]', ' ', texto)

    # Eliminar espacios múltiples
    texto = re.sub(r'\s+', ' ', texto)

    # Separar en palabras y filtrar
    palabras = texto.split()

    # Filtrar: stopwords, palabras muy cortas, palabras muy largas
    palabras_filtradas = [
        palabra for palabra in palabras
        if palabra not in stop_words_custom
        and len(palabra) > 3  # Mínimo 4 caracteres
        and len(palabra) < 20  # Máximo 20 caracteres (evita basura)
        and palabra.isalpha()  # Solo letras
    ]

    return " ".join(palabras_filtradas)


# Aplicar limpieza
print("\n🧹 Iniciando limpieza de textos...")
print("=" * 60)

ruta_csv = f"{path_proyecto}/datos_economicos_manabi.csv"

if os.path.exists(ruta_csv):
    df = pd.read_csv(ruta_csv)

    print(f"📊 Procesando {len(df)} registros...")

    # Aplicar limpieza mejorada
    df['texto_limpio'] = df['texto'].apply(limpiar_texto_mejorado)

    # Eliminar registros donde el texto limpio quedó muy corto (menos de 5 palabras)
    df['num_palabras'] = df['texto_limpio'].apply(lambda x: len(x.split()))
    df_filtrado = df[df['num_palabras'] >= 5].copy()

    eliminados = len(df) - len(df_filtrado)

    print(f"✅ Limpieza completada")
    print(f"  • Registros originales: {len(df)}")
    print(f"  • Registros eliminados (texto muy corto): {eliminados}")
    print(f"  • Registros válidos: {len(df_filtrado)}")

    # Guardar datos procesados
    ruta_procesado = f"{path_proyecto}/datos_manabi_procesados.csv"
    df_filtrado.to_csv(ruta_procesado, index=False, encoding='utf-8-sig')

    print(f"\n💾 Archivo procesado guardado")
    print(f"📁 Ubicación: datos_manabi_procesados.csv")

    # Comparación antes/después
    print("\n📝 COMPARACIÓN (Original vs Limpio):")
    print("-" * 60)
    muestra = df_filtrado.iloc[0]
    print(f"Original ({len(muestra['texto'])} caracteres):")
    print(f"{muestra['texto'][:200]}...\n")
    print(f"Limpio ({len(muestra['texto_limpio'])} caracteres):")
    print(f"{muestra['texto_limpio'][:200]}...")

    # Estadísticas de limpieza
    print(f"\n📊 Estadísticas de palabras:")
    print(f"  • Promedio de palabras por texto: {df_filtrado['num_palabras'].mean():.1f}")
    print(f"  • Mediana: {df_filtrado['num_palabras'].median():.0f}")
    print(f"  • Mínimo: {df_filtrado['num_palabras'].min()}")
    print(f"  • Máximo: {df_filtrado['num_palabras'].max()}")
else:
    print("❌ No se encontró el archivo de datos crudos")


🧹 Iniciando limpieza de textos...
📊 Procesando 12190 registros...
✅ Limpieza completada
  • Registros originales: 12190
  • Registros eliminados (texto muy corto): 10
  • Registros válidos: 12180

💾 Archivo procesado guardado
📁 Ubicación: datos_manabi_procesados.csv

📝 COMPARACIÓN (Original vs Limpio):
------------------------------------------------------------
Original (215 caracteres):
Construyen el mall más grande del mundo: costará 100 millones de dólares y tendrá 180 tiendas - El Cronista. Construyen el mall más grande del mundo: costará 100 millones de dólares y tendrá 180 tiend...

Limpio (141 caracteres):
construyen mall grande mundo costará millones dólares tiendas cronista construyen mall grande mundo costará millones dólares tiendas cronista...

📊 Estadísticas de palabras:
  • Promedio de palabras por texto: 17.2
  • Mediana: 16
  • Mínimo: 6
  • Máximo: 50


In [ ]:
# =================================================================
# CLASIFICADOR DE SENTIMIENTO
# =================================================================

def clasificador_alto_espectro_mejorado(texto):
    """
    Clasificador de sentimiento mejorado con más categorías
    Basado en raíces de palabras para mayor cobertura
    """
    texto = str(texto).lower()

    # RAÍCES NEGATIVAS EXPANDIDAS
    raices_negativas = [
        # Crisis económica
        'crisi', 'recesión', 'depresión', 'colapso', 'caíd', 'caer',
        'descens', 'baj', 'declin', 'deterior', 'derrumb',

        # Empleo y pobreza
        'desemple', 'despid', 'cesant', 'subemple', 'precari',
        'pobreza', 'pobres', 'miseria', 'hambre', 'escacez', 'escasez',
        'carenc', 'necesid', 'indigenc',

        # Violencia e inseguridad
        'violen', 'asesin', 'murder', 'killer', 'extorsio', 'vacuna',
        'secuestr', 'asalt', 'robo', 'delincuen', 'insegur', 'miedo',
        'narco', 'sicari', 'atenta', 'masacr',

        # Nombres específicos de bandas (Manabí)
        'chone', 'lobo', 'tiguero', 'latin', 'king',

        # Cierres y quiebras
        'clausur', 'cerr', 'cierr', 'quiebr', 'bancarrot', 'liquid',
        'suspend', 'paraliz', 'paro', 'huelg',

        # Problemas financieros
        'deud', 'morator', 'impag', 'incumpl', 'déficit', 'perdid', 'pérdid',

        # Desastres naturales/clima
        'lluvia', 'invierno', 'devastador', 'destru', 'destrucción',
        'terremoto', 'sismo', 'inundación', 'plaga', 'sequía',
        'afecta', 'dañ', 'catástrof',

        # Económico negativo
        'inflación', 'carestía', 'costo', 'encarec', 'sube', 'aument' + ' precio',
        'reducc', 'disminu', 'contracc',

        # Problemas sociales
        'corrupción', 'corrupto', 'fraude', 'ilegal', 'ilícit',
        'abandon', 'descuid', 'olvid'
    ]

    # RAÍCES POSITIVAS EXPANDIDAS
    raices_positivas = [
        # Crecimiento económico
        'crecimient', 'crecer', 'aumenta', 'aument', 'increment', 'sube',
        'alza', 'bonanza', 'prosper', 'auge', 'boom', 'mejora', 'mejor',
        'recupera', 'reactiva', 'repunt',

        # Empleo
        'emple', 'contrata', 'plazas', 'trabajo', 'oportunidad',

        # Comercio y negocios
        'comerci', 'venta', 'negoci', 'empresa', 'apertura', 'inaugura',
        'invers', 'capital', 'financiamiento', 'crédito', 'préstamo',

        # Producción
        'produc', 'cosecha', 'export', 'rendimiento', 'productiv',

        # Turismo
        'turism', 'turist', 'visit', 'hotel', 'gastronom', 'feriad',

        # Desarrollo
        'desarroll', 'proyect', 'obra', 'infraestructur', 'construc',
        'modern', 'tecnolog', 'innova', 'digital',

        # Apoyo gubernamental
        'gobierno', 'estado', 'ministeri', 'entrega', 'beneficia',
        'subsidio', 'ayuda', 'apoyo', 'programa', 'plan',

        # Positivo general
        'éxit', 'logro', 'gana', 'triunf', 'victoria', 'premio',
        'reconoc', 'destaca', 'sobresale', 'lidera', 'impulsa',
        'potencia', 'fortalec', 'consolid', 'millones', 'ingresos'
    ]

    # RAÍCES NEUTRAS (para desambiguar)
    raices_neutras = [
        'reunión', 'reunió', 'encuentr', 'sesión', 'asamblea',
        'anunci', 'inform', 'comunic', 'declar', 'manifest',
        'estudi', 'investig', 'analiz', 'revis'
    ]

    # Contar ocurrencias
    score_negativo = sum(1 for raiz in raices_negativas if raiz in texto)
    score_positivo = sum(1 for raiz in raices_positivas if raiz in texto)
    score_neutro = sum(1 for raiz in raices_neutras if raiz in texto)

    # LÓGICA DE CLASIFICACIÓN

    # Si hay balance entre positivo y negativo → Neutro
    if abs(score_positivo - score_negativo) <= 1 and score_neutro > 0:
        return 'Neutro'

    # Priorizar lo negativo (crisis, violencia, etc.)
    if score_negativo > score_positivo:
        return 'Negativo'

    # Si hay más positivo
    elif score_positivo > score_negativo:
        return 'Positivo'

    # Si no hay ninguna señal clara
    else:
        return 'Neutro'

# Aplicar clasificación
print("\n Aplicando clasificación de sentimiento...")
print("=" * 60)

ruta_procesado = f"{path_proyecto}/datos_manabi_procesados.csv"

if os.path.exists(ruta_procesado):
    df = pd.read_csv(ruta_procesado)

    print(f"📊 Clasificando {len(df)} registros...")

    # Aplicar clasificador
    df['sentimiento'] = df['texto_limpio'].apply(clasificador_alto_espectro_mejorado)

    # Guardar dataset final
    ruta_final = f"{path_proyecto}/dataset_etiquetado_tesis.csv"
    df.to_csv(ruta_final, index=False, encoding='utf-8-sig')

    print("\n Clasificación completada")
    print(f" Dataset final guardado: dataset_etiquetado_tesis.csv")

    print("\n DISTRIBUCIÓN DE SENTIMIENTOS:")
    print("=" * 60)
    distribucion = df['sentimiento'].value_counts()
    total = len(df)

    for sentimiento, count in distribucion.items():
        porcentaje = (count / total) * 100
        print(f"  • {sentimiento}: {count} ({porcentaje:.1f}%)")

    # Análisis temporal del sentimiento
    df['año'] = pd.to_datetime(df['fecha']).dt.year
    print("\n📅 Sentimiento por año:")
    print("-" * 60)

    for año in sorted(df['año'].unique()):
        df_año = df[df['año'] == año]
        dist_año = df_año['sentimiento'].value_counts()

        print(f"\n{año} ({len(df_año)} noticias):")
        for sent in ['Negativo', 'Neutro', 'Positivo']:
            if sent in dist_año:
                count = dist_año[sent]
                pct = (count / len(df_año)) * 100
                print(f"  {sent}: {count} ({pct:.1f}%)")

    # Ejemplos de cada sentimiento
    print("\n📰 EJEMPLOS POR SENTIMIENTO:")
    print("=" * 60)

    for sentimiento in ['Positivo', 'Neutro', 'Negativo']:
        ejemplos = df[df['sentimiento'] == sentimiento].head(2)
        print(f"\n{sentimiento.upper()}:")
        for idx, row in ejemplos.iterrows():
            print(f"  • [{row['fuente']}] {row['texto'][:100]}...")

else:
    print("❌ No se encontró el archivo procesado")


 Aplicando clasificación de sentimiento...
📊 Clasificando 12180 registros...

 Clasificación completada
 Dataset final guardado: dataset_etiquetado_tesis.csv

 DISTRIBUCIÓN DE SENTIMIENTOS:
  • Positivo: 5287 (43.4%)
  • Neutro: 5149 (42.3%)
  • Negativo: 1744 (14.3%)

📅 Sentimiento por año:
------------------------------------------------------------

2020 (509 noticias):
  Negativo: 75 (14.7%)
  Neutro: 229 (45.0%)
  Positivo: 205 (40.3%)

2021 (820 noticias):
  Negativo: 106 (12.9%)
  Neutro: 341 (41.6%)
  Positivo: 373 (45.5%)

2022 (1065 noticias):
  Negativo: 118 (11.1%)
  Neutro: 440 (41.3%)
  Positivo: 507 (47.6%)

2023 (1336 noticias):
  Negativo: 162 (12.1%)
  Neutro: 587 (43.9%)
  Positivo: 587 (43.9%)

2024 (2193 noticias):
  Negativo: 329 (15.0%)
  Neutro: 898 (40.9%)
  Positivo: 966 (44.0%)

2025 (4831 noticias):
  Negativo: 726 (15.0%)
  Neutro: 2043 (42.3%)
  Positivo: 2062 (42.7%)

2026 (1426 noticias):
  Negativo: 228 (16.0%)
  Neutro: 611 (42.8%)
  Positivo: 587 (41